# Módulo 9 · Clase 2 — Limpieza y Preparación de Datos

**Proyecto Final: Sistema Inteligente de Predicción y Exposición de Modelos**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Auditoría de calidad de datos |
| 2 | Tratamiento de duplicados y nulos |
| 3 | Outliers: detección y decisión |
| 4 | Encoding y transformaciones |
| 5 | Feature engineering y guardado final |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Cargar datos crudos de la clase anterior
df_raw = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
print(f"Dataset crudo cargado: {df_raw.shape}")
df_raw.head(3)

---
## 1. Auditoría de calidad de datos

In [ ]:
def auditoria_calidad(df):
    """Informe completo de calidad de un DataFrame."""
    print(f"{'AUDITORÍA DE CALIDAD':=^60}")
    print(f"  Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}")
    print(f"  Memoria: {df.memory_usage(deep=True).sum()/1024:.0f} KB")

    print(f"\n{'Columna':30} {'Tipo':10} {'Nulos':>6} {'%':>6} {'Únicos':>8} {'Min':>10} {'Max':>10}")
    print('-' * 85)
    for col in df.columns:
        nulos   = df[col].isna().sum()
        pct     = nulos / len(df) * 100
        unicos  = df[col].nunique()
        if pd.api.types.is_numeric_dtype(df[col]):
            mn, mx = df[col].min(), df[col].max()
        else:
            mn, mx = '-', '-'
        print(f"  {col:28} {str(df[col].dtype):10} {nulos:6} {pct:6.1f}% {unicos:8} {str(mn):>10} {str(mx):>10}")

    print(f"\n  Duplicados exactos: {df.duplicated().sum()}")
    print(f"  Correlación máxima entre features (excl. diagonal):")
    num = df.select_dtypes('number')
    corr = num.corr().abs()
    np.fill_diagonal(corr.values, 0)
    idx = np.unravel_index(corr.values.argmax(), corr.shape)
    print(f"    {corr.columns[idx[0]]} ↔ {corr.columns[idx[1]]}: {corr.values.max():.3f}")

auditoria_calidad(df_raw)

---
## 2. Tratamiento de duplicados y nulos

In [ ]:
df = df_raw.copy()
print(f"Dataset original: {df.shape}")

# --- Duplicados ---
n_dup = df.duplicated().sum()
print(f"\nDuplicados encontrados: {n_dup}")
if n_dup > 0:
    # Mostrar un ejemplo
    print("Ejemplo de duplicados:")
    print(df[df.duplicated(keep=False)].head(4))
    # En datos de vinos, los duplicados son valores FÍSICAMENTE POSIBLES
    # (dos vinos pueden tener exactamente las mismas características)
    # Decisión: eliminarlos para evitar data leakage en validación cruzada
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"\nTras eliminar duplicados: {df.shape} ({n_dup} filas eliminadas)")

# --- Nulos ---
n_nulos = df.isna().sum().sum()
print(f"\nValores nulos totales: {n_nulos}")
if n_nulos > 0:
    print(df.isna().sum()[df.isna().sum() > 0])
else:
    print("✓ Sin valores nulos — no es necesaria imputación")

# Simular nulos para demo (en un proyecto real vendrían de los datos)
print("\nSimulando nulos para demostrar estrategias de imputación:")
df_con_nulos = df.copy()
np.random.seed(42)
for col in ['pH', 'alcohol', 'sulphates']:
    idx_nulos = np.random.choice(len(df_con_nulos), size=20, replace=False)
    df_con_nulos.loc[idx_nulos, col] = np.nan

print(f"  Nulos añadidos: {df_con_nulos.isna().sum().sum()}")

# Estrategias de imputación
print("\nEstrategias de imputación:")
for estrategia, func in [
    ('Mediana',  lambda s: s.fillna(s.median())),
    ('Media',    lambda s: s.fillna(s.mean())),
    ('Moda',     lambda s: s.fillna(s.mode()[0])),
]:
    df_temp = df_con_nulos.copy()
    for col in ['pH', 'alcohol', 'sulphates']:
        df_temp[col] = func(df_temp[col])
    print(f"  {estrategia}: {df_temp.isna().sum().sum()} nulos restantes")

# En nuestro dataset original no hay nulos → continuamos sin imputar
print(f"\nDataset final sin duplicados ni nulos: {df.shape}")

---
## 3. Outliers: detección y decisión

In [ ]:
# Métodos de detección de outliers

def detectar_outliers_iqr(serie, factor=1.5):
    """Regla IQR: outlier si < Q1 - 1.5·IQR o > Q3 + 1.5·IQR"""
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    return (serie < Q1 - factor*IQR) | (serie > Q3 + factor*IQR)

def detectar_outliers_zscore(serie, umbral=3.0):
    """Z-score: outlier si |z| > umbral"""
    return (np.abs(stats.zscore(serie.dropna())) > umbral)

cols_num = df.select_dtypes('number').columns.drop('quality')

print(f"{'Columna':30} {'Outliers IQR':>14} {'% IQR':>8} {'Outliers Z>3':>14}")
print('-' * 70)
for col in cols_num:
    n_iqr = detectar_outliers_iqr(df[col]).sum()
    pct_iqr = n_iqr / len(df) * 100
    n_z   = detectar_outliers_zscore(df[col]).sum()
    print(f"  {col:28} {n_iqr:14} {pct_iqr:8.1f}% {n_z:14}")

# Visualizar outliers con boxplots
fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for ax, col in zip(axes.ravel(), cols_num):
    ax.boxplot(df[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col, fontsize=9)
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.3, axis='y')
axes.ravel()[-1].set_visible(False)
plt.suptitle('Outliers por variable (boxplot IQR)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Decisión sobre outliers: clipping (Winsorización)
# Recortamos al percentil 1 y 99 para no perder datos

df_limpio = df.copy()

cols_a_clipear = ['fixed acidity', 'volatile acidity', 'citric acid',
                   'residual sugar', 'chlorides', 'free sulfur dioxide',
                   'total sulfur dioxide', 'sulphates']

limites = {}
for col in cols_a_clipear:
    p1, p99 = df_limpio[col].quantile(0.01), df_limpio[col].quantile(0.99)
    antes = detectar_outliers_iqr(df_limpio[col]).sum()
    df_limpio[col] = df_limpio[col].clip(p1, p99)
    despues = detectar_outliers_iqr(df_limpio[col]).sum()
    limites[col] = (p1, p99)
    print(f"  {col:30}: [{p1:.4f}, {p99:.4f}] — outliers IQR: {antes}→{despues}")

print(f"\n✓ Clipping aplicado en {len(cols_a_clipear)} columnas")
print(f"  Las columnas density, pH, alcohol tienen outliers razonables → sin clipear")

---
## 4. Encoding y transformaciones

In [ ]:
# Encoding de la variable 'tipo' (tinto/blanco)

# Opción 1: Label Encoding (para modelos que aceptan ordinales)
df_limpio['tipo_encoded'] = (df_limpio['tipo'] == 'tinto').astype(int)  # 1=tinto, 0=blanco

# Opción 2: One-Hot Encoding
df_ohe = pd.get_dummies(df_limpio, columns=['tipo'], prefix='tipo', drop_first=False)
print("Columnas tras One-Hot Encoding:")
print([c for c in df_ohe.columns if 'tipo' in c])

# Transformaciones de distribución
# Algunas variables están muy sesgadas → log transform las normaliza
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
cols_sesgadas = ['residual sugar', 'chlorides', 'free sulfur dioxide']

for i, col in enumerate(cols_sesgadas):
    # Original
    axes[0, i].hist(df_limpio[col], bins=40, color='coral', edgecolor='white')
    sk = df_limpio[col].skew()
    axes[0, i].set_title(f'{col}\nskewness={sk:.2f}')
    axes[0, i].set_xlabel('Original')

    # Log-transform
    log_vals = np.log1p(df_limpio[col])  # log(1+x) para evitar log(0)
    sk_log = log_vals.skew()
    axes[1, i].hist(log_vals, bins=40, color='steelblue', edgecolor='white')
    axes[1, i].set_title(f'log1p({col})\nskewness={sk_log:.2f}')
    axes[1, i].set_xlabel('Log-transformada')

plt.suptitle('Efecto de la transformación logarítmica', fontsize=13)
plt.tight_layout()
plt.show()

# Aplicar log-transform a las variables más sesgadas
for col in cols_sesgadas:
    df_limpio[f'{col}_log'] = np.log1p(df_limpio[col])

print("\nLog-transforms añadidas:", [f'{c}_log' for c in cols_sesgadas])

---
## 5. Feature Engineering y guardado final

In [ ]:
# Feature Engineering: crear variables derivadas con significado enológico

# Ratio dióxido de azufre libre/total: indica cuánto SO2 está 'activo'
df_limpio['so2_ratio'] = (
    df_limpio['free sulfur dioxide'] / (df_limpio['total sulfur dioxide'] + 1e-8)
)

# Índice de acidez: combina acidez fija y volátil
df_limpio['acid_index'] = df_limpio['fixed acidity'] - df_limpio['volatile acidity']

# Densidad relativa: cómo se aleja la densidad del agua (1.0)
df_limpio['density_diff'] = np.abs(df_limpio['density'] - 1.0)

# Variable objetivo adicional: clasificación binaria
# Buen vino = calidad >= 7
df_limpio['buen_vino'] = (df_limpio['quality'] >= 7).astype(int)

print(f"Features nuevas: so2_ratio, acid_index, density_diff")
print(f"Target binario:  buen_vino (1 si quality >= 7)")
print(f"\nDistribución de 'buen_vino':")
print(df_limpio['buen_vino'].value_counts())
print(f"  Proporción buenos: {df_limpio['buen_vino'].mean():.2%}")

# Correlación de features con el target
cols_features = [c for c in df_limpio.select_dtypes('number').columns
                 if c not in ['quality', 'buen_vino', 'tipo_encoded']]
corr_target = df_limpio[cols_features + ['buen_vino']].corr()['buen_vino'].drop('buen_vino')
corr_target = corr_target.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colores = ['green' if v > 0 else 'red' for v in corr_target.values]
ax.barh(corr_target.index, corr_target.values, color=colores, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Correlación con buen_vino'); ax.set_title('Features vs target')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
import json
from datetime import datetime

# Definir columnas finales para el modelo
FEATURES_ML = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol',
    'so2_ratio', 'acid_index', 'density_diff',
    'tipo_encoded'
]
TARGET_REG = 'quality'
TARGET_CLF = 'buen_vino'

# Dataset final limpio
df_final = df_limpio[FEATURES_ML + [TARGET_REG, TARGET_CLF, 'tipo']].copy()

print(f"Dataset procesado: {df_final.shape}")
print(f"Features: {FEATURES_ML}")
print(f"Targets: {TARGET_REG} (regresión), {TARGET_CLF} (clasificación)")

# Guardar
df_final.to_parquet('data/processed/vinos_procesado.parquet', index=False)
df_final.to_csv('data/processed/vinos_procesado.csv', index=False)

# Metadatos del preprocesado
meta_proc = {
    'fecha': datetime.now().isoformat(),
    'filas_raw':       int(df_raw.shape[0]),
    'filas_procesado': int(df_final.shape[0]),
    'features':        FEATURES_ML,
    'target_regresion': TARGET_REG,
    'target_clasificacion': TARGET_CLF,
    'limites_clipping': {k: [float(v[0]), float(v[1])] for k,v in limites.items()},
    'proporcion_buenos': float(df_final[TARGET_CLF].mean()),
    'pasos': [
        'Eliminar duplicados',
        'Clipping p1-p99 en variables sesgadas',
        'Label encoding de tipo',
        'Log-transform en residual sugar, chlorides, free SO2',
        'Feature engineering: so2_ratio, acid_index, density_diff',
        'Crear target binario buen_vino (quality >= 7)'
    ]
}
with open('data/processed/preprocesado_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta_proc, f, indent=2, ensure_ascii=False)

print("\n✓ Dataset procesado guardado en data/processed/")

---
## Práctica

### Ejercicio 1 — Mapa de correlaciones

**Enunciado:**
1. Calcula la matriz de correlación de Pearson para todas las features numéricas
2. Plótala como heatmap con `seaborn.heatmap`
3. Identifica los 3 pares de variables más correlacionadas
4. ¿Alguna podría eliminarse por ser redundante?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
cols_num = df.select_dtypes('number').columns.drop(['quality', 'buen_vino'])
corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))  # ocultar triángulo superior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Mapa de correlaciones — Wine Quality', fontsize=14)
plt.tight_layout()
plt.show()

# Top 3 pares
corr_abs = corr.abs()
np.fill_diagonal(corr_abs.values, 0)
pares = []
for _ in range(3):
    idx = np.unravel_index(corr_abs.values.argmax(), corr_abs.shape)
    c1, c2 = corr_abs.columns[idx[0]], corr_abs.columns[idx[1]]
    pares.append((c1, c2, corr.loc[c1, c2]))
    corr_abs.loc[c1, c2] = 0
    corr_abs.loc[c2, c1] = 0

print("Top 3 correlaciones:")
for c1, c2, r in pares:
    print(f"  {c1} ↔ {c2}: r={r:.3f}")
```
</details>

---
### Ejercicio 2 — Comparativa de distribuciones por calidad

**Enunciado:** ¿Cómo difieren las features entre vinos de baja y alta calidad?

1. Para cada feature numérica, calcula la media por grupo (buen_vino=0 vs 1)
2. Calcula el t-test entre grupos (`scipy.stats.ttest_ind`)
3. Muestra las features con diferencia estadísticamente significativa (p < 0.05)
4. Ordénalas por tamaño del efecto (diferencia de medias normalizada)

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
features = [c for c in df.select_dtypes('number').columns
            if c not in ['quality', 'buen_vino']]

buenos    = df[df['buen_vino'] == 1]
no_buenos = df[df['buen_vino'] == 0]

resultados = []
for feat in features:
    b  = buenos[feat].dropna()
    nb = no_buenos[feat].dropna()
    t, p = stats.ttest_ind(b, nb)
    efecto = (b.mean() - nb.mean()) / (df[feat].std() + 1e-8)  # Cohen's d simplif.
    resultados.append({'feature': feat, 'media_buenos': b.mean(),
                       'media_no_buenos': nb.mean(), 'p_value': p, 'efecto': efecto})

df_res = pd.DataFrame(resultados).sort_values('efecto', key=abs, ascending=False)
sig = df_res[df_res['p_value'] < 0.05]

print(f"Features significativas (p<0.05): {len(sig)}/{len(features)}")
print(sig[['feature','media_buenos','media_no_buenos','p_value','efecto']].round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colores = ['green' if v > 0 else 'red' for v in sig['efecto']]
ax.barh(sig['feature'], sig['efecto'], color=colores, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Efecto (positivo = más en buenos, negativo = más en no buenos)')
ax.set_title('Tamaño del efecto por feature')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
```
</details>

---
### Ejercicio 3 — Pipeline de preprocesado reproducible

**Enunciado:** Encapsula todo el preprocesado en una función pura:

```python
def preprocesar(df_raw, config) -> pd.DataFrame:
    ...
```

Donde `config` es un dict con los parámetros (umbral de calidad, columnas a clipear, etc.).
La función debe ser **idempotente**: llamarla dos veces con el mismo input da el mismo resultado.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import pandas as pd
import numpy as np

CONFIG_DEFAULT = {
    'umbral_calidad_buena': 7,
    'cols_clip': ['fixed acidity', 'volatile acidity', 'citric acid',
                  'residual sugar', 'chlorides', 'free sulfur dioxide',
                  'total sulfur dioxide', 'sulphates'],
    'percentil_clip': (0.01, 0.99),
    'features_finales': [
        'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
        'chlorides', 'free sulfur dioxide', 'total sulfur dioxide',
        'density', 'pH', 'sulphates', 'alcohol',
        'so2_ratio', 'acid_index', 'density_diff', 'tipo_encoded'
    ]
}

def preprocesar(df_raw, config=None):
    if config is None:
        config = CONFIG_DEFAULT
    df = df_raw.copy()

    # 1. Eliminar duplicados
    df = df.drop_duplicates().reset_index(drop=True)

    # 2. Clipping
    p_low, p_high = config['percentil_clip']
    for col in config['cols_clip']:
        if col in df.columns:
            lo, hi = df[col].quantile(p_low), df[col].quantile(p_high)
            df[col] = df[col].clip(lo, hi)

    # 3. Encoding
    df['tipo_encoded'] = (df['tipo'] == 'tinto').astype(int)

    # 4. Feature engineering
    df['so2_ratio']    = df['free sulfur dioxide'] / (df['total sulfur dioxide'] + 1e-8)
    df['acid_index']   = df['fixed acidity'] - df['volatile acidity']
    df['density_diff'] = np.abs(df['density'] - 1.0)

    # 5. Targets
    df['buen_vino'] = (df['quality'] >= config['umbral_calidad_buena']).astype(int)

    return df[config['features_finales'] + ['quality', 'buen_vino', 'tipo']]

# Test de idempotencia
df_raw = pd.read_parquet('data/raw/vinos_combinados_raw.parquet')
df_p1 = preprocesar(df_raw)
df_p2 = preprocesar(df_raw)  # segunda ejecución
assert (df_p1.values == df_p2.values).all(), "¡No es idempotente!"
print(f"✓ Función idempotente. Resultado: {df_p1.shape}")
```
</details>

---
### Ejercicio 4 — Análisis de sesgo por tipo de vino

**Enunciado:** ¿Tienen los vinos tintos y blancos la misma distribución de calidad?

1. Compara la distribución de `quality` entre tintos y blancos
2. Aplica un test de Mann-Whitney U (`scipy.stats.mannwhitneyu`) para ver si son iguales
3. ¿Habría que estratificar el train/test split por tipo de vino?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_parquet('data/processed/vinos_procesado.parquet')

tintos  = df[df['tipo'] == 'tinto']['quality']
blancos = df[df['tipo'] == 'blanco']['quality']

u_stat, p_value = stats.mannwhitneyu(tintos, blancos, alternative='two-sided')
print(f"Mann-Whitney U: U={u_stat:.0f}, p={p_value:.6f}")
print(f"{'Distribuciones significativamente distintas' if p_value < 0.05 else 'No hay diferencia significativa'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for tipo, color in [('tinto','firebrick'), ('blanco','steelblue')]:
    datos = df[df['tipo']==tipo]['quality']
    datos.value_counts().sort_index().plot(
        kind='bar', ax=axes[0], color=color, alpha=0.6, label=tipo)
    datos.plot(kind='kde', ax=axes[1], color=color, lw=2, label=tipo)

axes[0].set_title('Distribución de calidad por tipo'); axes[0].legend()
axes[1].set_title('KDE de calidad'); axes[1].legend()
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nMedia calidad — Tinto: {tintos.mean():.2f}, Blanco: {blancos.mean():.2f}")
print(f"% buenos — Tinto: {(tintos>=7).mean():.2%}, Blanco: {(blancos>=7).mean():.2%}")
print("\n→ Con diferencias significativas, usar stratify=['buen_vino','tipo'] en train_test_split")
```
</details>

---
### Ejercicio 5 (reto) — Reporte HTML de calidad de datos

**Enunciado:** Genera un reporte HTML automático del dataset procesado:

1. Instala `ydata-profiling` (`pip install ydata-profiling`) o usa pandas + matplotlib
2. Si tienes ydata-profiling: `ProfileReport(df).to_file('docs/reporte_calidad.html')`
3. Si no: crea una función que genere un HTML con tabla de estadísticas + imágenes base64
4. El reporte debe incluir: shape, nulos, distribuciones, correlaciones

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import pandas as pd
import matplotlib.pyplot as plt
import base64, io
from pathlib import Path

def fig_a_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

def generar_reporte_html(df, ruta_salida='docs/reporte_calidad.html'):
    Path('docs').mkdir(exist_ok=True)
    imagenes = []

    # Histogramas
    cols_num = df.select_dtypes('number').columns.drop(['quality','buen_vino']).tolist()
    fig, axes = plt.subplots(3, 5, figsize=(18, 10))
    for ax, col in zip(axes.ravel(), cols_num):
        ax.hist(df[col], bins=30, color='steelblue', edgecolor='white')
        ax.set_title(col, fontsize=8); ax.grid(True, alpha=0.3)
    for ax in axes.ravel()[len(cols_num):]:
        ax.set_visible(False)
    fig.suptitle('Distribuciones')
    imagenes.append(('Distribuciones', fig_a_base64(fig)))

    # Estadísticas
    tabla_html = df.describe().round(3).to_html(classes='tabla')

    html = f"""
    <!DOCTYPE html><html lang="es"><head>
    <meta charset="UTF-8"><title>Reporte de Calidad</title>
    <style>body{{font-family:Arial;margin:30px}} .tabla{{border-collapse:collapse;width:100%}}
    .tabla td,.tabla th{{border:1px solid #ddd;padding:6px;font-size:12px}}
    .tabla th{{background:#4472C4;color:white}} h1,h2{{color:#333}}</style></head>
    <body><h1>Reporte de Calidad — Wine Quality Dataset</h1>
    <p>Filas: {df.shape[0]:,} | Columnas: {df.shape[1]} | Nulos: {df.isna().sum().sum()}</p>
    <h2>Estadísticas</h2>{tabla_html}
    {''.join(f"<h2>{t}</h2><img src='data:image/png;base64,{img}' style='max-width:100%'>" for t, img in imagenes)}
    </body></html>
    """

    Path(ruta_salida).write_text(html, encoding='utf-8')
    print(f"Reporte generado: {ruta_salida}")

df = pd.read_parquet('data/processed/vinos_procesado.parquet')
generar_reporte_html(df)
```
</details>

---
## Resumen

```python
# Pipeline de limpieza
df = df_raw.copy()
df = df.drop_duplicates().reset_index(drop=True)          # duplicados
df['col'] = df['col'].fillna(df['col'].median())           # nulos
df['col'] = df['col'].clip(df['col'].quantile(0.01),       # outliers
                            df['col'].quantile(0.99))

# Encoding
df['cat_encoded'] = (df['cat'] == 'valor').astype(int)     # binario
df = pd.get_dummies(df, columns=['cat'])                    # one-hot

# Transformaciones
df['col_log'] = np.log1p(df['col'])                        # log-transform

# Feature engineering
df['ratio'] = df['a'] / (df['b'] + 1e-8)
df['diff']  = df['a'] - df['b']

# Target binario
df['target_bin'] = (df['score'] >= umbral).astype(int)

# Guardar
df.to_parquet('data/processed/procesado.parquet', index=False)
```

## ¿Qué viene en la próxima clase?
En la **Clase 3** almacenaremos el dataset en una base de datos **SQL** (SQLite), aprenderemos a hacer consultas y usaremos SQLAlchemy para conectar Python con la BD.